# BlueprintGPT Room Planner Training

**For VS Code + Colab Extension**

This notebook trains the room planner transformer using Google Colab's GPU from within VS Code.

## Prerequisites
1. Install VS Code extension: **Colab Notebooks**
2. Open this notebook in VS Code
3. Select **Google Colab** as the kernel
4. VS Code will handle the connection to Colab's GPU automatically

## Architecture
- **Planner Model**: Predicts room order + centroids + adjacency  
- **Algorithmic Packer**: Converts guidance into geometry
- **Training**: ~40 epochs on T4 GPU (~30-45 min)

## Setup and Configuration

In [2]:
import json
import os
import shlex
import subprocess
from pathlib import Path

# Configuration - modify these if needed
USE_TEACHER_DATA = True       # Generate synthetic training data
TEACHER_CASES = 64            # Number of synthetic cases to generate
TRAINING_EPOCHS = 40          # Training epochs (increase if needed)
BATCH_SIZE = 64               # Reduce to 32 if you get CUDA memory errors

# Auto-detect environment: VS Code + Colab extension vs actual Google Colab
cwd = Path.cwd()

# Check if we're in the BlueprintGPT repo already (VS Code + Colab extension)
if (cwd / 'requirements.txt').exists() and (cwd / 'learned').exists():
    print('🏠 Detected: VS Code + Colab extension (local repo)')
    print(f'📁 Repository directory: {cwd}')
elif Path('/content/BlueprintGPT').exists():
    print('☁️  Detected: Google Colab (uploaded repo)')
    os.chdir('/content/BlueprintGPT')
    print(f'📁 Repository directory: /content/BlueprintGPT')
else:
    print('❌ BlueprintGPT repo not found!')
    print(f'   Current directory: {cwd}')
    print('   Expected files: requirements.txt, learned/ directory')
    print('   ')
    print('   Solutions:')
    print('   1. For VS Code + Colab: Open notebook from repo root')
    print('   2. For Google Colab: Upload repo to /content/BlueprintGPT')
    raise FileNotFoundError('BlueprintGPT repository not found')

# Paths (relative to repo root)
PLANNER_DATA_DIR = Path('learned/planner/data')
TEACHER_RECORDS = PLANNER_DATA_DIR / 'planner_teacher_records.jsonl'
PLANNER_RECORDS = PLANNER_DATA_DIR / 'planner_records.jsonl'
TRAIN_RECORDS = PLANNER_DATA_DIR / 'planner_train.jsonl'
VAL_RECORDS = PLANNER_DATA_DIR / 'planner_val.jsonl'
PROFILE_JSON = PLANNER_DATA_DIR / 'planner_profile.json'
CHECKPOINT_PATH = Path('learned/planner/checkpoints/room_planner.pt')
BENCHMARK_ROOT = Path('outputs/planner_benchmark_vscode')
BENCHMARK_JSON = BENCHMARK_ROOT / 'summary.json'

def run_cmd(args):
    """Execute command and show output"""
    cmd_str = ' '.join(shlex.quote(str(arg)) for arg in args)
    print(f'🔧 Running: {cmd_str}')
    result = subprocess.run([str(arg) for arg in args], capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ Error: {result.stderr}')
        raise subprocess.CalledProcessError(result.returncode, args)
    if result.stdout.strip():
        print(result.stdout)
    print('✅ Done\n')

# Verify we're in the right directory
print(f'📁 Working directory: {Path.cwd()}')
print(f'🔍 Using teacher data: {USE_TEACHER_DATA} ({TEACHER_CASES} cases)')
print(f'🎯 Training config: {TRAINING_EPOCHS} epochs, batch size {BATCH_SIZE}')
print(f'💾 Checkpoint will be saved to: {CHECKPOINT_PATH}')

# FIXED: Better file checking with absolute paths
current_dir = Path.cwd()
print(f'\n🔍 Verifying project files:')
print(f'   📁 Working in: {current_dir}')

# Check each file individually
files_to_check = {
    'requirements.txt': current_dir / 'requirements.txt',
    'planner training module': current_dir / 'learned' / 'planner' / 'train.py',
    'learned directory': current_dir / 'learned',
    'planner module': current_dir / 'learned' / 'planner' / '__init__.py'
}

missing_files = []
for name, path in files_to_check.items():
    if path.exists():
        print(f'   ✅ {name}: {path.relative_to(current_dir)}')
    else:
        print(f'   ❌ {name}: {path.relative_to(current_dir)} (MISSING)')
        missing_files.append(str(path.relative_to(current_dir)))

if missing_files:
    print(f'\n❌ Missing required files: {missing_files}')
    print('   Make sure you\'re in the BlueprintGPT repo root!')
    print('   Try: File → Open Folder → BlueprintGPT')
    raise FileNotFoundError(f'Required project files not found: {missing_files}')
else:
    print(f'\n✅ All project files found - ready to train!')

❌ BlueprintGPT repo not found!
   Current directory: /content
   Expected files: requirements.txt, learned/ directory
   
   Solutions:
   1. For VS Code + Colab: Open notebook from repo root
   2. For Google Colab: Upload repo to /content/BlueprintGPT


FileNotFoundError: BlueprintGPT repository not found

## Install Dependencies

This installs PyTorch and other required packages on the Colab runtime.

In [ ]:
# Upgrade pip first
run_cmd(['python', '-m', 'pip', 'install', '--upgrade', 'pip'])

# Install project dependencies
run_cmd(['pip', 'install', '-r', 'requirements.txt'])

# Verify PyTorch GPU is available
import torch
print(f'🚀 PyTorch version: {torch.__version__}')
print(f'🎮 CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🔥 GPU device: {torch.cuda.get_device_name(0)}')
    print(f'💾 GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  WARNING: CUDA not available, training will be slow on CPU!')

## Generate Synthetic Teacher Data

This creates diverse training examples from the algorithmic backend to improve room-program coverage.

In [ ]:
if USE_TEACHER_DATA:
    PLANNER_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f'🏗️ Generating {TEACHER_CASES} synthetic teacher records...')
    run_cmd([
        'python', '-m', 'learned.planner.synthesize_teacher_data',
        '--output', str(TEACHER_RECORDS),
        '--num-cases', str(TEACHER_CASES),
        '--seed', '42',
    ])
    
    # Show basic stats
    if TEACHER_RECORDS.exists():
        with open(TEACHER_RECORDS) as f:
            lines = f.readlines()
        print(f'✅ Generated {len(lines)} teacher records')
        
        # Show first record structure
        if lines:
            sample = json.loads(lines[0])
            print(f'📝 Sample record: {sample["plan_id"]} with {len(sample["rooms"])} rooms')
    else:
        print('❌ No teacher records generated')
else:
    print('⏭️ Skipping synthetic teacher data generation')

## Build Training Dataset

Combines Kaggle-derived records + synthetic teacher data → train/validation splits.

In [ ]:
print('📚 Building planner dataset...')

prepare_cmd = [
    'python', '-m', 'learned.planner.prepare_dataset',
    '--input-pattern', 'learned/data/kaggle_json/*.json',
    '--output', str(PLANNER_RECORDS),
    '--train-output', str(TRAIN_RECORDS),
    '--val-output', str(VAL_RECORDS),
    '--val-ratio', '0.2',
    '--seed', '42',
]

if USE_TEACHER_DATA and TEACHER_RECORDS.exists():
    prepare_cmd += ['--append-jsonl', str(TEACHER_RECORDS)]

run_cmd(prepare_cmd)

# Show dataset sizes
datasets = {'Combined': PLANNER_RECORDS, 'Training': TRAIN_RECORDS, 'Validation': VAL_RECORDS}
for name, path in datasets.items():
    if path.exists():
        with open(path) as f:
            count = len(f.readlines())
        print(f'📊 {name} dataset: {count} records')
    else:
        print(f'❌ {name} dataset not found at {path}')

## Profile Dataset Quality

Analyze room-program distribution to ensure training data balance.

In [ ]:
print('📋 Profiling dataset composition...')

run_cmd([
    'python', '-m', 'learned.planner.profile_dataset',
    '--input', str(PLANNER_RECORDS),
    '--top-programs', '12',
    '--output-json', str(PROFILE_JSON),
])

# Display profile summary
if PROFILE_JSON.exists():
    profile = json.loads(PROFILE_JSON.read_text(encoding='utf-8'))
    
    print('\n📈 Dataset Analysis:')
    print(f"   Total records: {profile['record_count']}")
    print(f"   Teacher records: {profile['teacher_record_count']}")
    print(f"   Base records: {profile['base_record_count']}")
    print(f"   Average rooms per layout: {profile['avg_room_count']}")
    
    print('\n🏠 Room Type Distribution:')
    for room_type, count in list(profile['room_type_counts'].items())[:8]:
        print(f"   {room_type}: {count}")
    
    print('\n📐 Most Common Room Programs:')
    for i, prog in enumerate(profile['top_programs'][:6]):
        print(f"   {i+1}. {prog['program']} ({prog['count']} layouts)")
        
    # Quality checks
    teacher_ratio = profile['teacher_record_count'] / profile['record_count']
    print(f'\n🎯 Quality Indicators:')
    print(f"   Teacher data ratio: {teacher_ratio:.1%} {'✅' if teacher_ratio > 0.15 else '⚠️'}")
    print(f"   Program diversity: {len(profile['top_programs'])} unique programs {'✅' if len(profile['top_programs']) > 10 else '⚠️'}")
else:
    print('❌ Profile not generated')

## Train Planner Model

**This is the main training phase** (~30-45 minutes on T4 GPU)

⏱️ **Expected timeline:**
- Setup: ~2 min
- Training: ~30-40 min (40 epochs)
- Total: ~45 min

📊 **Monitor these metrics:**
- **Train loss**: Should decrease to <1.5
- **Val loss**: Should follow train loss (not diverge)
- **Gap**: Val loss - train loss should be <0.5


In [ ]:
print(f'🚀 Starting planner training ({TRAINING_EPOCHS} epochs)...')
print(f'💾 Checkpoint will be saved to: {CHECKPOINT_PATH}')
print('\n📊 Training progress will show:')
print('   - Epoch progress bars')
print('   - Train/Val loss per epoch')
print('   - Best checkpoint updates')

# Ensure checkpoint directory exists
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

train_cmd = [
    'python', '-m', 'learned.planner.train',
    '--train', str(TRAIN_RECORDS),
    '--val', str(VAL_RECORDS),
    '--save', str(CHECKPOINT_PATH),
    '--epochs', str(TRAINING_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--lr', '3e-4',
    '--weight-decay', '1e-4',
    '--d-model', '192',
    '--n-heads', '6',
    '--n-layers', '6',
    '--d-ff', '384',
    '--dropout', '0.1',
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
]

print(f'🎮 Using device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

try:
    run_cmd(train_cmd)
    print('🎉 Training completed successfully!')
    
    if CHECKPOINT_PATH.exists():
        size_mb = CHECKPOINT_PATH.stat().st_size / 1024 / 1024
        print(f'💾 Checkpoint saved: {CHECKPOINT_PATH} ({size_mb:.1f} MB)')
    else:
        print('❌ Checkpoint not found after training')
        
except subprocess.CalledProcessError as e:
    print(f'❌ Training failed: {e}')
    print('\n🔧 Troubleshooting:')
    print('   - If CUDA OOM: Reduce BATCH_SIZE to 32')
    print('   - If convergence issues: Increase TRAINING_EPOCHS to 60')
    print('   - Check GPU availability above')
    raise

## Benchmark Trained Model

Compare planner vs algorithmic on test cases to validate quality.

🎯 **Success criteria:**
- Planner wins ≥50% of test cases
- Average scores within 0.05 of baseline
- No major quality regressions

In [ ]:
if not CHECKPOINT_PATH.exists():
    print('❌ Checkpoint not found - skipping benchmark')
else:
    print('🏁 Running planner vs algorithmic benchmark...')
    
    # Set checkpoint environment variable
    os.environ['BLUEPRINTGPT_PLANNER_CHECKPOINT'] = str(CHECKPOINT_PATH)
    
    # Ensure benchmark output directory exists
    BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
    
    benchmark_cmd = [
        'python', '-m', 'learned.planner.benchmark',
        '--output-root', str(BENCHMARK_ROOT),
        '--output-json', str(BENCHMARK_JSON),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    ]
    
    try:
        run_cmd(benchmark_cmd)
        print('✅ Benchmark completed!')
        
        if BENCHMARK_JSON.exists():
            benchmark_data = json.loads(BENCHMARK_JSON.read_text(encoding='utf-8'))
            summary = benchmark_data.get('summary', {})
            
            print('\n📊 Benchmark Results:')
            planner_avg = summary.get('planner_avg_score', 0)
            algo_avg = summary.get('algorithmic_avg_score', 0)
            planner_wins = summary.get('planner_better_count', 0)
            algo_wins = summary.get('algorithmic_better_count', 0)
            total_cases = planner_wins + algo_wins
            
            print(f'   📈 Planner average score: {planner_avg:.3f}')
            print(f'   📈 Algorithmic average score: {algo_avg:.3f}')
            print(f'   🏆 Planner wins: {planner_wins}/{total_cases} ({planner_wins/total_cases*100:.0f}%)')
            print(f'   🏆 Algorithmic wins: {algo_wins}/{total_cases} ({algo_wins/total_cases*100:.0f}%)')
            
            # Quality assessment
            if planner_wins >= total_cases * 0.5 and abs(planner_avg - algo_avg) < 0.1:
                print('\n🎉 EXCELLENT: Planner is ready for deployment!')
                deployment_status = '✅ DEPLOY'
            elif planner_wins >= total_cases * 0.4:
                print('\n⚠️  MIXED: Planner is competitive but needs monitoring')
                deployment_status = '⚠️ MONITOR'
            else:
                print('\n❌ POOR: Planner needs more training')
                deployment_status = '❌ RETRAIN'
                
            print(f'\n🚀 Deployment recommendation: {deployment_status}')            
            print(f'\n📋 Full benchmark results saved to: {BENCHMARK_JSON}')
            
        else:
            print('❌ Benchmark results not found')
            
    except subprocess.CalledProcessError as e:
        print(f'❌ Benchmark failed: {e}')
        print('⚠️  This might indicate issues with the trained model')

## Deployment Instructions

**The checkpoint is now saved in your local repo!** Here's how to enable it:

In [ ]:
print('🎯 DEPLOYMENT CHECKLIST:')
print()
print('1. ✅ Checkpoint file:')
if CHECKPOINT_PATH.exists():
    size_mb = CHECKPOINT_PATH.stat().st_size / 1024 / 1024
    print(f'   💾 {CHECKPOINT_PATH} ({size_mb:.1f} MB)')
    print('   ✅ Ready to use!')
else:
    print('   ❌ Checkpoint not found - training may have failed')
    
print('\n2. 🔧 Environment configuration:')
print('   Add to your .env file:')
print(f'   BLUEPRINTGPT_PLANNER_CHECKPOINT={CHECKPOINT_PATH}')
print()
print('   For controlled rollout (recommended):')   
print('   BLUEPRINT_AUTO_CORE_BACKEND=planner_if_available')
print()
print('   OR for explicit planner mode:')
print('   BLUEPRINT_BACKEND_MODE=planner')

print('\n3. 🚀 Restart your server:')
print('   uvicorn api.server:app --reload --port 8000')

print('\n4. ✅ Verify it\'s working:')
print('   - Open: http://localhost:8000/ui/')
print('   - Settings panel should show: "Planner: ✅ Available"')
print('   - Test prompt: "Design a 3BHK apartment"')
print('   - Response should mention: "Engine: Planner-guided"')

print('\n5. 📊 Optional - Run local benchmark:')
print(f'   .venv/Scripts/python.exe -m learned.planner.benchmark \\')
print(f'     --output-root outputs/planner_local_benchmark \\')
print(f'     --output-json outputs/planner_local_benchmark/summary.json')

if BENCHMARK_JSON.exists():
    benchmark_data = json.loads(BENCHMARK_JSON.read_text(encoding='utf-8'))
    summary = benchmark_data.get('summary', {})
    planner_wins = summary.get('planner_better_count', 0)
    total_cases = planner_wins + summary.get('algorithmic_better_count', 0)
    
    print(f'\n📈 Training results: Planner won {planner_wins}/{total_cases} cases')
    
    if planner_wins >= total_cases * 0.5:
        print('🎉 RECOMMENDATION: Enable planner with planner_if_available')
    else:
        print('⚠️  RECOMMENDATION: Keep algorithmic as default for now')
else:
    print('\n⚠️  No benchmark data available - test manually before deployment')
    
print('\n🎊 Training complete! Your planner model is ready to use.')

## Summary & Next Steps

🎉 **Congratulations!** You've successfully trained a room planner transformer.

### What you've accomplished:
- ✅ Generated synthetic teacher data for better coverage
- ✅ Built balanced training dataset (Kaggle + synthetic)
- ✅ Trained planner model on GPU
- ✅ Benchmarked against algorithmic baseline
- ✅ Created deployable checkpoint

### Architecture achieved:
This implements the exact **encoder-decoder + deterministic constraints** architecture recommended:
- **Text/Spec Encoder**: NL → structured room requirements  
- **Planner Model**: Room order + adjacency + centroid priors
- **Algorithmic Packer**: Robust geometry generation
- **Hard Constraints**: Always enforced at inference
- **Repair/Ranking**: Quality assurance pipeline intact

### Performance expectations:
- **Good checkpoint**: Wins 50-70% of test cases, similar average scores
- **Excellent checkpoint**: Wins 70%+ with better average scores
- **Needs work**: Wins <40% or Much lower average scores

### Troubleshooting:
- **High validation loss**: Try more training epochs (60-80)
- **Poor benchmark**: Generate more teacher data (--num-cases 128)
- **Memory errors**: Reduce batch size to 32 or 16
- **Slow convergence**: Adjust learning rate to 2e-4 or 1e-4

**Ready for production?** Follow the deployment checklist above! 🚀